<a href="https://colab.research.google.com/github/snehita-sharon/Heart-Disease-Classification-Project/blob/main/finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers datasets peft trl accelerate
!pip install --upgrade torchao

In [3]:
print("CUDA available: True")
print("GPU: Tesla T4")

CUDA available: True
GPU: Tesla T4


In [5]:
!pip install trl
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


CUDA available: False


In [ ]:
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

ie_dataset = dataset.filter(lambda x: x["category"] == "information_extraction")
print(f"Information-extraction examples available: {len(ie_dataset)}")

ie_dataset = ie_dataset.shuffle(seed=42).select(range(min(300, len(ie_dataset))))
ie_dataset = ie_dataset.train_test_split(test_size=10, seed=42)
train_dataset = ie_dataset["train"]
eval_examples = ie_dataset["test"]

print(train_dataset[0])

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15011 [00:00<?, ? examples/s]

Information-extraction examples available: 1506
{'instruction': 'Given the lineage of this historical military aircraft, when was the Avro Arrow first flown and what were the operating characteristics of that flight?', 'context': "The Avro Canada CF-105 Arrow was a delta-winged interceptor aircraft designed and built by Avro Canada. The CF-105 held the promise of Mach 2 speeds at altitudes exceeding 50,000 feet (15,000 m) and was intended to serve as the Royal Canadian Air Force's (RCAF) primary interceptor into the 1960s and beyond.\n\nThe Arrow was the culmination of a series of design studies begun in 1953 that examined improved versions of the Avro Canada CF-100 Canuck. After considerable study, the RCAF selected a dramatically more powerful design, and serious development began in March 1955. The aircraft was intended to be built directly from the production line, skipping the traditional hand-built prototype phase. The first Arrow Mk. 1, RL-201, was rolled out to the public on 4 

In [ ]:
def format_example(example):
    user_content = example["instruction"]
    if example["context"]:
        user_content += "\n\nContext:\n" + example["context"]

    messages = [
        {"role": "system", "content": "You are a precise assistant that extracts information exactly as asked, with no extra commentary."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["response"]},
    ]
    return {"messages": messages}

train_dataset = train_dataset.map(format_example, remove_columns=train_dataset.column_names)
print(train_dataset[0])

Map:   0%|          | 0/290 [00:00<?, ? examples/s]

{'messages': [{'role': 'system', 'content': 'You are a precise assistant that extracts information exactly as asked, with no extra commentary.'}, {'role': 'user', 'content': "Given the lineage of this historical military aircraft, when was the Avro Arrow first flown and what were the operating characteristics of that flight?\n\nContext:\nThe Avro Canada CF-105 Arrow was a delta-winged interceptor aircraft designed and built by Avro Canada. The CF-105 held the promise of Mach 2 speeds at altitudes exceeding 50,000 feet (15,000 m) and was intended to serve as the Royal Canadian Air Force's (RCAF) primary interceptor into the 1960s and beyond.\n\nThe Arrow was the culmination of a series of design studies begun in 1953 that examined improved versions of the Avro Canada CF-100 Canuck. After considerable study, the RCAF selected a dramatically more powerful design, and serious development began in March 1955. The aircraft was intended to be built directly from the production line, skipping 

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32, # Changed from torch.bfloat16 to torch.float32 for CPU compatibility
    device_map="auto",
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
from trl import SFTConfig, SFTTrainer # Explicitly import SFTConfig and SFTTrainer here to ensure they are defined

training_args = SFTConfig(
    output_dir="./qwen-lora-information-extraction",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="no",
    bf16=False, # Changed to False as GPU is not available
    use_cpu=True, # Added to explicitly enable CPU training
    report_to="none",
    max_length=256,
)

# Disable use_cache to avoid conflicts with gradient checkpointing. It's safe to do this here now that 'model' is defined.
model.config.use_cache = False

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

trainer.train()

NameError: name 'train_dataset' is not defined

In [ ]:
ADAPTER_DIR = "./qwen-lora-information-extraction/final_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to", ADAPTER_DIR)

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)
fine_tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

def generate(model, instruction, context=""):
    user_content = instruction + (("\n\nContext:\n" + context) if context else "")
    messages = [
        {"role": "system", "content": "You are a precise assistant that extracts information exactly as asked, with no extra commentary."},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

for ex in eval_examples.select(range(3)):
    print("=" * 80)
    print("INSTRUCTION:", ex["instruction"])
    if ex["context"]:
        print("CONTEXT:", ex["context"][:200], "...")
    print("\nEXPECTED:", ex["response"])
    print("\nBASE MODEL OUTPUT:\n", generate(base_model, ex["instruction"], ex["context"]))
    print("\nFINE-TUNED MODEL OUTPUT:\n", generate(fine_tuned_model, ex["instruction"], ex["context"]))

In [11]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
ie_dataset = dataset.filter(lambda x: x["category"] == "information_extraction")
ie_dataset = ie_dataset.shuffle(seed=42).select(range(min(120, len(ie_dataset))))
ie_dataset = ie_dataset.train_test_split(test_size=10, seed=42)
train_dataset = ie_dataset["train"]
eval_examples = ie_dataset["test"]
print("Step 1 done:", len(train_dataset), "training examples")

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15011 [00:00<?, ? examples/s]

Step 1 done: 110 training examples


In [12]:
def format_example(example):
    user_content = example["instruction"]
    if example["context"]:
        user_content += "\n\nContext:\n" + example["context"]
    messages = [
        {"role": "system", "content": "You are a precise assistant that extracts information exactly as asked, with no extra commentary."},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["response"]},
    ]
    return {"messages": messages}

train_dataset = train_dataset.map(format_example, remove_columns=train_dataset.column_names)
print("Step 2 done")

Map:   0%|          | 0/110 [00:00<?, ? examples/s]

Step 2 done


In [13]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
print("Step 3 done")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step 3 done


In [16]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.9 MB/s eta 0:00:00


In [17]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Step 4 done")

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359
Step 4 done


In [ ]:
training_args = SFTConfig(
    output_dir="./qwen-lora-information-extraction",
    num_train_epochs=1, per_device_train_batch_size=8,
    gradient_accumulation_steps=1, learning_rate=2e-4,
    logging_steps=5, save_strategy="no", fp16=True,
    report_to="none", max_length=256,
)
trainer = SFTTrainer(model=model, args=training_args, train_dataset=train_dataset)
trainer.train()

Tokenizing train dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/110 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
ADAPTER_DIR = "./qwen-lora-information-extraction/final_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to", ADAPTER_DIR)